# Skills Marketplace

This notebook demonstrates discovering and using a third-party skill from the [Skills Marketplace](https://skillsmp.com) — a community-curated catalog of reusable agent skills.

We will:
1. **Discover** the `slack-gif-creator` skill via the SkillsMP REST API
2. **Install** it from its [GitHub source](https://github.com/anthropics/skills/tree/main/skills/slack-gif-creator) into `.agents/skills/`
3. **Run** an `LLMAgent` that activates the skill to generate an animated GIF

The skill bundles PIL animation utilities and Slack-specific knowledge for creating optimised GIFs. It is sourced from Anthropic's public skills repository and illustrates the core value of a marketplace: a skill authored once by anyone is instantly usable by any agent, anywhere.

> **Note:** Run this notebook from within `more-examples/ch06/` so the installed skill is discovered automatically as a project-scoped skill.

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPI by uncommenting the cell below.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama installed on your local machine and have its LLM hosting service running. To download Ollama, follow the instructions found on this page: https://ollama.com/download. After downloading and installing Ollama, you can start a service by opening a terminal and running the command `ollama serve`.

## Skill Dependencies

The `slack-gif-creator` skill uses Python's PIL library and a few imaging helpers to generate animated GIFs. Install them into the running Jupyter kernel before executing the agent cells.

In [ ]:
!uv pip install pillow imageio imageio-ffmpeg numpy

## Discovering the Skill

The SkillsMP REST API lets you search the catalog programmatically. Here we search for `slack-gif-creator` and locate the entry that points to Anthropic's source repository.

In [ ]:
import json
import urllib.request

SKILLSMP_API = "https://skillsmp.com/api/v1"

with urllib.request.urlopen(
    f"{SKILLSMP_API}/skills/search?q=slack-gif-creator&limit=5&sortBy=stars",
) as resp:
    data = json.loads(resp.read())

skills = data["data"]["skills"]

print(f"Found {data['data']['pagination']['total']} results. Top matches:\n")
for s in skills:
    print(f"  {s['name']} — {s['githubUrl']}")

## Installing the Skill

Skills live in GitHub repositories. We download the repo as a zip archive and extract the `slack-gif-creator` directory into `.agents/skills/` — the project-scoped skill location that `LLMAgent` discovers automatically.

We install from Anthropic's canonical source at [`anthropics/skills`](https://github.com/anthropics/skills/tree/main/skills/slack-gif-creator), which is the repository the marketplace entries mirror.

In [ ]:
import io
import zipfile
from pathlib import Path

REPO = "anthropics/skills"
SKILL_SUBPATH = "skills/slack-gif-creator"

zip_url = f"https://github.com/{REPO}/archive/refs/heads/main.zip"
skill_dir = Path(".agents/skills/slack-gif-creator")
skill_dir.mkdir(parents=True, exist_ok=True)

with urllib.request.urlopen(zip_url) as resp:
    zip_bytes = resp.read()

prefix = f"skills-main/{SKILL_SUBPATH}/"

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    for name in zf.namelist():
        if name.startswith(prefix) and not name.endswith("/"):
            rel = name[len(prefix) :]
            dest = skill_dir / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(zf.read(name))

print(f"Installed to: {skill_dir.resolve()}")
print("\nFiles:")
for f in sorted(skill_dir.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(skill_dir)}")

## Inspecting the Skill

Before running the agent, let's confirm the skill was discovered and take a look at its `SKILL.md`.

In [ ]:
import logging

from llm_agents_from_scratch import TOOLS_FOR_SKILL_RESOURCES, LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging

enable_console_logging(logging.INFO)

llm = OllamaLLM(model="qwen3:14b", think=False)
agent = LLMAgent(llm=llm, tools=TOOLS_FOR_SKILL_RESOURCES)

task = Task(instruction="placeholder")
handler = LLMAgent.TaskHandler(llm_agent=agent, task=task)
skill = handler.skills["slack-gif-creator"]

print(f"Name:      {skill.frontmatter.name}")
print(f"Scope:     {skill.scope}")
print(f"Skill dir: {skill_dir.resolve()}")
print("\nResources:")
for r in skill.resources:
    print(f"  - {r}")

## Generating a GIF

Now we run the agent with the `slack-gif-creator` skill. The agent will:

1. Activate the skill via `UseSkillTool` to read its instructions
2. Write PIL animation code based on the prompt
3. Execute it with `PythonInterpreterTool` using `code=` and `cwd=skill_dir` so the bundled `core/` modules are importable
4. Save the result as `output.gif` inside the skill directory

In [ ]:
OUTPUT_GIF = skill_dir.resolve() / "output.gif"

result = await agent.run_with_skill(
    "slack-gif-creator",
    prompt=(
        "Create a spinning star emoji GIF (128x128 px, optimized for Slack)."
        f" Save the output to {OUTPUT_GIF}."
    ),
    max_steps=10,
)
print(result)

## Result

In [ ]:
from IPython.display import Image

Image(filename=str(OUTPUT_GIF))